# Comparação de cenários de dados — modelo vencedor (AG 03/Exp2_padrao)

Qual conjunto de features entrega melhor recall no pipeline de produção, fixando os hiperparâmetros vencedores do AG?

**B_strict** = pipeline de produção padrão (12 features: contínuas + ordinais + PAI_AUSENTE)

**B_plus** = B_strict + PNTARDIO + HISTPERDAFETAL + PRIMIPARA + FAIXAETAMAE OHE (16 features)

Ambos usam `HISTGB_PARAMS` de `src/constants.py` = **AG 03/Exp2_padrao** (`learning_rate=0.20, max_iter=100, max_leaf_nodes=127, min_samples_leaf=20, l2_regularization=0.0`) e threshold 0.40.

## Hipóteses

Dois aspectos do `build_pipeline()` diferem do caminho de pesquisa (parquet + `apply_scenario`):

**H1 — StandardScaler é desnecessário para HistGB**  
`HistGradientBoostingClassifier` é baseado em árvores e cria histogramas internos de binagem.
Ele é invariante a escala — o `StandardScaler` no pipeline de produção não prejudica nem ajuda.

**H2 — ESTCIVMAE, RACACORMAE e SEXO não adicionam sinal preditivo**  
Essas features estão no parquet (Cenário B de pesquisa, 29 features) mas são dropadas como
leakage pelo `clean_dataset()` no caminho de produção (12 features).
A hipótese é que o sinal preditivo para prematuridade está concentrado nas features
obstétricas e de pré-natal — e que status civil, raça e sexo do bebê não acrescentam
poder discriminativo além do que o modelo já captura.

A comparação abaixo verifica essas hipóteses empiricamente.

## Configuração do experimento

- **B_strict**: pipeline de produção (`src/pipeline_factory.build_pipeline()`), 12 features após FeatureEngineer + StandardScaler
- **B_plus**: B_strict + PNTARDIO, HISTPERDAFETAL, PRIMIPARA, FAIXAETAMAE OHE (16 features)
- Mesmo modelo base: `HistGradientBoostingClassifier` com `Exp1_conservador` do AG 03
- Threshold operacional: 0.40
- Referência: `best_model_calibrated.pkl` da Fase 1 (treinado com Cenário B de pesquisa, 29 features)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, fbeta_score, precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

from src.cleaning import clean_dataset
from src.transformers import FeatureEngineer
from src.pipeline_factory import build_pipeline
from src.constants import (
    DEFAULT_THRESHOLD, RANDOM_STATE, HISTGB_PARAMS,
    NUMERIC_COLUMNS, ESCMAE2010_ORDINAL_COLUMN, KOTELCHUCK_ORDINAL_COLUMN,
)

print('HISTGB_PARAMS (AG 03/Exp2_padrao):', HISTGB_PARAMS)

## 1. Dados

In [2]:
df_raw = pd.read_parquet('../data/df_model_raw.parquet')
df_clean = clean_dataset(df_raw)

y = df_clean['PREMATURO']
X = df_clean.drop(columns=['PREMATURO'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Prevalência prematuro (treino): {y_train.mean():.3%}')
print(f'Colunas disponíveis: {X_train.columns.tolist()}')

Train: (552344, 12) | Test: (138087, 12)
Prevalência prematuro (treino): 10.201%
Colunas disponíveis: ['IDADEMAE', 'ESCMAE2010', 'QTDGESTANT', 'QTDPARTNOR', 'QTDPARTCES', 'QTDFILVIVO', 'QTDFILMORT', 'MESPRENAT', 'KOTELCHUCK', 'LATITUDE', 'LONGITUDE', 'PAI_AUSENTE']


## 2. Definição das pipelines

In [ ]:
def build_pipeline_b_strict(random_state: int = RANDOM_STATE) -> Pipeline:
    """12 features: ordinais + contínuas + PAI_AUSENTE. Replica o Cenário B de produção."""
    numeric_cols = list(NUMERIC_COLUMNS) + [ESCMAE2010_ORDINAL_COLUMN, KOTELCHUCK_ORDINAL_COLUMN]
    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', StandardScaler(), numeric_cols),
            ('passthrough', 'passthrough', ['PAI_AUSENTE']),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ('feature_engineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('histgradientboostingclassifier', HistGradientBoostingClassifier(
            random_state=random_state, **HISTGB_PARAMS
        )),
    ])


class _FeatureEngineerBPlus(FeatureEngineer):
    """Estende FeatureEngineer com as flags derivadas do pipeline do Caê."""

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        out = super().transform(X)
        out['PNTARDIO']      = (out['MESPRENAT'].fillna(99) > 4).astype(int)
        out['HISTPERDAFETAL'] = (out['QTDFILMORT'].fillna(0) > 0).astype(int)
        out['PRIMIPARA']     = ((out['QTDPARTNOR'].fillna(0) + out['QTDPARTCES'].fillna(0)) == 0).astype(int)
        out['FAIXAETAMAE']   = pd.cut(
            out['IDADEMAE'],
            bins=[10, 20, 25, 30, 35, 40, 61],
            labels=[0, 1, 2, 3, 4, 5],
            right=False,
        ).astype(float)
        return out


def build_pipeline_b_plus(random_state: int = RANDOM_STATE) -> Pipeline:
    """16 features: B_strict + PNTARDIO + HISTPERDAFETAL + PRIMIPARA + FAIXAETAMAE."""
    numeric_cols = list(NUMERIC_COLUMNS) + [ESCMAE2010_ORDINAL_COLUMN, KOTELCHUCK_ORDINAL_COLUMN]
    flag_cols    = ['PAI_AUSENTE', 'PNTARDIO', 'HISTPERDAFETAL', 'PRIMIPARA', 'FAIXAETAMAE']
    preprocessor = ColumnTransformer(
        transformers=[
            ('numeric',     StandardScaler(), numeric_cols),
            ('passthrough', 'passthrough',    flag_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ('feature_engineer', _FeatureEngineerBPlus()),
        ('preprocessor', preprocessor),
        ('histgradientboostingclassifier', HistGradientBoostingClassifier(
            random_state=random_state, **HISTGB_PARAMS
        )),
    ])

## 3. Treinar ambos

In [ ]:
weights   = compute_sample_weight('balanced', y_train)
fit_kwargs = {'histgradientboostingclassifier__sample_weight': weights}

print('Treinando B_strict + calibração...')
cal_strict = CalibratedClassifierCV(build_pipeline_b_strict(), method='isotonic', cv=3)
cal_strict.fit(X_train, y_train, sample_weight=weights)

print('Treinando B_plus + calibração...')
cal_plus = CalibratedClassifierCV(build_pipeline_b_plus(), method='isotonic', cv=3)
cal_plus.fit(X_train, y_train, sample_weight=weights)

print('Pronto.')

## 4. Avaliar @ threshold 0.40

In [ ]:
def evaluate(model, X_te, y_te, threshold=DEFAULT_THRESHOLD, name='', features=None):
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_te == 1)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    fp = int(((pred == 1) & (y_te == 0)).sum())
    return {
        'modelo':    name,
        'recall':    round(recall_score(y_te, pred), 4),
        'f2':        round(fbeta_score(y_te, pred, beta=2), 4),
        'precisão':  round(precision_score(y_te, pred, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y_te, prob), 4),
        'TP': tp, 'FN': fn, 'FP': fp,
        'features': features,
    }

with open('../results/metrics/best_model_operational_metrics.json') as f:
    bl = json.load(f)['metrics']

resultados = pd.DataFrame([
    {'modelo': 'Baseline Fase 1 (pkl calibrado)',
     'recall': round(bl['recall'], 4), 'f2': round(bl['f2'], 4),
     'precisão': round(bl['precision'], 4), 'roc_auc': round(bl['roc_auc'], 4),
     'TP': bl['tp'], 'FN': bl['fn'], 'FP': bl['fp'], 'features': 12},
    evaluate(cal_strict, X_test, y_test, name='B_strict + AG params (calibrado)', features=12),
    evaluate(cal_plus,   X_test, y_test, name='B_plus  + AG params (calibrado)', features=16),
])

print(resultados.to_string(index=False))

## 5. Referência da Fase 1 (baseline calibrado)

In [6]:
with open('../results/metrics/best_model_operational_metrics.json') as f:
    baseline = json.load(f)

m = baseline['metrics']
print('=== Referência Fase 1 (best_model_calibrated.pkl) ===')
print(f"  recall:   {m['recall']:.4f}")
print(f"  f2:       {m['f2']:.4f}")
print(f"  precisão: {m['precision']:.4f}")
print(f"  roc_auc:  {m['roc_auc']:.4f}")
print(f"  TP: {m['tp']}  FN: {m['fn']}  FP: {m['fp']}")

=== Referência Fase 1 (best_model_calibrated.pkl) ===
  recall:   0.8135
  f2:       0.3823
  precisão: 0.1225
  roc_auc:  0.6535
  TP: 11459  FN: 2627  FP: 82083


---

## Conclusão

### Resultado com AG 03/Exp2_padrao + calibração isotônica

| modelo | features | recall | f2 | FN | FP | ΔFN vs baseline | ΔFP vs baseline |
|---|---|---|---|---|---|---|---|
| Baseline Fase 1 (pkl calibrado) | 12 | 0.8135 | 0.3823 | 2 627 | 82 083 | — | — |
| **B_strict + AG params** | **12** | **0.8279** | 0.3805 | **2 424** | 85 255 | **−203** | +3.9% |
| B_plus + AG params | 16 | 0.8244 | 0.3809 | 2 473 | 84 465 | −154 | +2.9% |

### Decisão: B_strict vence

**B_strict evita 203 prematuros não sinalizados a mais que o baseline** — B_plus evita apenas 154.

O B_plus reduz 790 falsos positivos em relação ao B_strict (84 465 vs 85 255), mas essa economia de FP vem ao custo de 49 FN a mais. Como a **prioridade clínica é minimizar FN** (não sinalizar um prematuro é mais grave que alarmar um caso de baixo risco), B_strict é superior.

Ambos passam nos critérios clínicos: recall ≥ 0.80 e ΔFP ≤ +15% vs baseline.

### A decisão de remover as features do B_plus foi correta

Com os hiperparâmetros da Fase 1 (RandomizedSearch), B_plus já perdia para B_strict.  
Com os hiperparâmetros do AG (Exp2_padrao), B_plus continua perdendo no critério prioritário (FN).  
As 4 features extras (PNTARDIO, HISTPERDAFETAL, PRIMIPARA, FAIXAETAMAE) não agregam sinal preditivo útil para o HistGB neste problema.

### Pipeline de produção confirmado

```
df_model_raw.parquet
  → clean_dataset()       # 12 colunas após remoção de leakage
  → build_pipeline()      # FeatureEngineer + StandardScaler + HistGB (AG params)
  → CalibratedClassifierCV(method='isotonic', cv=3)
  → model_phase2.pkl      ← artefato de produção
```

Nenhuma alteração necessária em `src/pipeline_factory.py`.